In [22]:
import requests

r = requests.get("https://mapping.gleif.org/api/v2/isin-lei/d6b9e6ba-8322-4010-b6f0-c5870d28e83a/download", stream=True)

# Open the file in write-binary mode
with open("isin-lei.zip", "wb") as f:
    # Iterate over the response in chunks
    for chunk in r.iter_content(chunk_size=8192):
        # Write each chunk to the file
        if chunk:  # filter out keep-alive new chunks
            f.write(chunk)

In [10]:
import re
from urllib.parse import urlsplit


def is_adls_uri(uri: str) -> bool:
    format = uri.split("//")
    if re.match(r"abfs:", format[0], re.IGNORECASE):
        return True
    return False


def is_s3_uri(uri: str) -> bool:
    format = uri.split("//")
    if re.match(r"s3[na]?:", format[0], re.IGNORECASE):
        return True
    return False


def convert_to_adls_uri(uri: str, account_name: str) -> str:
    if not account_name:
        raise Exception("Account name is missing.")
    if not uri:
        raise Exception("URI is missing.")
    if not is_adls_uri(uri):
        raise Exception("Not a valid ADLS URI.")

    parsed = urlsplit(uri)
    container = parsed.netloc
    path = parsed.path.lstrip("/")

    return f"abfs://{container}@{account_name}.dfs.core.windows.net/{path}"


def convert_to_s3_uri(uri: str) -> str:
    if not uri:
        raise Exception("URI is missing.")
    if not is_s3_uri(uri):
        raise Exception(f"'{uri}' is not a valid S3 URI.")

    parsed = urlsplit(uri)
    bucket = parsed.netloc
    key = parsed.path.lstrip("/")

    return f"s3a://{bucket}/{key}"


convert_to_s3_uri("s3://raw/path")

's3a://raw/path'

In [5]:
import os

from pyiceberg.catalog import load_catalog

AWS_REGION = os.environ.get("AWS_REGION")
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY")

catalog = load_catalog(
    "docs",
    **{
        "uri": "thrift://localhost:9083",
        "s3.access-key-id": AWS_ACCESS_KEY_ID,
        "s3.secret-access-key": AWS_SECRET_ACCESS_KEY,
        "s3.region": AWS_REGION,
        # "s3.access-key-id": "test",
        # "s3.secret-access-key": "testSecret",
        # "s3.region": "us-east-1",
        # "s3.endpoint": "http://localhost:9000",
    },
)

# catalog.list_namespaces()
# catalog.create_namespace("hallo")
# catalog.load_table("curated.exchange").scan().to_pandas()
catalog.list_tables("curated")

[('curated', 'exchange'),
 ('curated', 'security'),
 ('curated', 'security_quote'),
 ('curated', 'fundamental'),
 ('curated', 'entity_isin'),
 ('curated', 'entity'),
 ('curated', 'security_quote_performance')]

In [3]:
from pyiceberg.schema import Schema
from pyiceberg.types import (
    TimestampType,
    FloatType,
    DoubleType,
    StringType,
    NestedField,
    StructType,
)

schema = Schema(
    NestedField(field_id=1, name="datetime", field_type=TimestampType(), required=True),
    NestedField(field_id=2, name="symbol", field_type=StringType(), required=True),
    NestedField(field_id=3, name="bid", field_type=FloatType(), required=False),
    NestedField(field_id=4, name="ask", field_type=DoubleType(), required=False),
    NestedField(
        field_id=5,
        name="details",
        field_type=StructType(
            NestedField(field_id=4, name="created_by", field_type=StringType(), required=False),
        ),
        required=False,
    ),
)

from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.transforms import DayTransform

partition_spec = PartitionSpec(
    PartitionField(source_id=1, field_id=1000, transform=DayTransform(), name="datetime_day")
)

from pyiceberg.table.sorting import SortOrder, SortField
from pyiceberg.transforms import IdentityTransform

# Sort on the symbol
sort_order = SortOrder(SortField(source_id=2, transform=IdentityTransform()))

catalog.create_table(
    identifier="test.nein",
    schema=schema,
    # location="s3a://warehouse/iceberg",
    partition_spec=partition_spec,
    sort_order=sort_order,
)

In [5]:
catalog.list_tables("curated")

[]

In [11]:
import os

# P("../dags/spark").resolve().relative_to()

P(os.getcwd()).parent.resolve()

PosixPath('/Users/stefanjakobbauer/Development/projects/unique-stocks/services/pipelines')

In [1]:
# import os
import zipfile


def unzip_file(path: str, delete_zip_file=False):
    """
    Unzip a file on a local file system.
    """

    with zipfile.ZipFile(path, allowZip64=True) as zip_archive:
        file_to_unzip = zip_archive.filelist[0]

        unzipped_file_path = zip_archive.extract(file_to_unzip, path="new")

    return unzipped_file_path


unzipped_file = unzip_file("isin-lei.zip")

In [2]:
unzipped_file

'new/lei-isin-20240123T080530.csv'

In [25]:
import duckdb

a = duckdb.read_csv(unzipped_file)
a.write_csv("sadf.csv.gz", compression="gzip")

In [21]:
from utils.file.unzip import unzip_file

unzip_file("isin-lei.zip")

AirflowConfigException: Cannot use relative path: `sqlite:///testing/airflow/airflow.db` to connect to sqlite. Please use absolute path such as `sqlite:////tmp/airflow.db`.

In [19]:
import duckdb

a = duckdb.read_csv("nasdf.csv.gz", header=True, compression="gzip")
a.write_csv("nasasdfasdfasdfsadfdf2.csv.gz", compression="gzip")

In [4]:
import zipfile


def unzip_file(zip_filepath, dest_dir):
    with zipfile.ZipFile(zip_filepath, "r") as zip_ref:
        zip_ref.extractall(dest_dir)


# Usage
unzip_file("isin-lei.zip", "isin-lei")

In [5]:
with zipfile.ZipFile("isin-lei.zip", allowZip64=True) as zip_archive:
    file_to_unzip = zip_archive.filelist[0]

    unzipped_file_path = zip_archive.extract(file_to_unzip, path="sadfs")

In [4]:
import gzip

In [5]:
import polars as pl

pl.read_csv("hallo.csv.gz")

LEI,ISIN
str,str
"""001GPB6A9XPE8X…","""US3158052262"""
"""00EHHQ2ZHDCFXJ…","""US92204Q1031"""
"""00KLB2PFTM3060…","""US4138382027"""
"""00KLB2PFTM3060…","""US4138385749"""
"""01ERPZV3DOLNXY…","""US531554AA10"""
"""01ERPZV3DOLNXY…","""US531554AB92"""
"""01ERPZV3DOLNXY…","""US531554AC75"""
"""01ERPZV3DOLNXY…","""US531554AD58"""
"""01ERPZV3DOLNXY…","""US531554AE32"""


In [20]:
import gzip
from pathlib import Path as P
import os
from uuid import uuid4


def compress_with_gzip(path: str, chunk_size=8192, delete_source_file=True) -> str:
    """
    Compress a file on a local file system using gzip.
    The compressed file is saved in the same directory as the source file.

    Args:
        path (str): Path to local uncompressed file.
        chunk_size (int, optional): _description_. Defaults to 8,192.
        delete_source_file (bool, optional): _description_. Defaults to True.

    Returns:
        str: Absolute path to local compressed file.
    """

    filename = uuid4().hex
    ext = P(path).suffix.replace(".", "")

    zip_path = P(path).parent / f"{filename}.{ext}.gz"

    with open(path, "rb") as f_in, gzip.open(zip_path, "wb") as f_out:
        while True:
            content = f_in.read(chunk_size)
            if not content:
                break
            f_out.write(content)

    if delete_source_file:
        os.remove(path)

    return zip_path.absolute().as_posix()


compress_with_gzip("lei-isin-20240123T080530.csv", delete_source_file=False)

'/Users/stefanjakobbauer/Development/projects/unique-stocks/services/pipelines/testing/91cb063ee6684e9e88bc23b7a396b9eb.csv.gz'

In [1]:
import duckdb
import polars as pl

pl.read_csv(
    "/Users/stefanjakobbauer/Development/projects/unique-stocks/services/pipelines/testing/91cb063ee6684e9e88bc23b7a396b9eb.csv.gz"
).head(5)

FileNotFoundError: No such file or directory (os error 2): ...rojects/unique-stocks/services/pipelines/testing/91cb063ee6684e9e88bc23b7a396b9eb.csv.gz